In [5]:
"""Validación cuantitativa de FARMSAR (solo CAE, sin Otsu).
 
Pensado para usarse tras tu script de entrenamiento: necesita en el espacio de
nombres Conv1DAutoencoder, entrenar_cae_por_clase y predict_pixels.
 
Flujo:
  1. aplanar_test            -> X, Y, pid del split test
  2. parcelas_fiables        -> parcelas de test con >75 % de píxeles coherentes
                                (usando los CAE entrenados con train)
  3. infectar                -> permuta etiquetas de un % de parcelas fiables
  4. entrenar_farmsar        -> 10 pasadas de purga sobre el conjunto infectado
  5. relabel_por_parcela     -> regla del 75 % sobre la clase candidata
  6. metricas                -> recall y precisión por parcela
"""
import numpy as np
import random
import torch
import os 
import joblib
import csv
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier


In [6]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, input_channels=2, seq_len=61, bottleneck_dim=1):
        super().__init__()

        self.input_channels = input_channels
        self.seq_len = seq_len

        # ============================================================
        # ENCODER CONVOLUCIONAL
        # ============================================================
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, stride=1, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 61 -> 28

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 28 -> 12

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 12 -> 5
        )

        self.flatten = nn.Flatten()

        # ============================================================
        # FC ENCODER (Bottleneck)
        # ============================================================
        self.fc_enc = nn.Sequential(
            nn.Linear(256 * 5, 128),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, bottleneck_dim),
            nn.ELU()  # Activación ELU en el espacio latente (Tabla 2 del paper)
        )

        # ============================================================
        # FC DECODER (Adaptable dinámicamente)
        # ============================================================
        self.fc_dec = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 128),
            nn.ELU(),
            nn.Linear(128, input_channels * seq_len),  # 2x61=122 o 4x61=244
        )

    def forward(self, x):
        x_enc = self.encoder(x)                            # (B, 256, 5)
        x_flat = self.flatten(x_enc)                       # (B, 1280)
        z = self.fc_enc(x_flat)                            # (B, bottleneck_dim)
        x_dec = self.fc_dec(z)                             # (B, C * T)
        x_dec = x_dec.view(-1, self.input_channels, self.seq_len) # (B, C, T)
        return x_dec

In [7]:
UMBRAL = 0.75
N_MIN_PARCELAS = 5

def cargar_modelos(
    path_modelos, 
    clases, 
    pasada=10, 
    input_channels=2, 
    seq_len=61, 
    bottleneck_dim=1, 
    device="cpu",
    infestado=False  # <- Nuevo parámetro para elegir el tipo de archivo
):
    """Carga los CAE guardados, permitiendo elegir entre modelos normales o infestados (_crop_)."""
    modelos = {}
    
    for c in clases:
        # Construimos el nombre del archivo dinámicamente según el parámetro
        if infestado:
            nombre = f"cae_pasada_{pasada}_crop_{c}.pth"
        else:
            nombre = f"cae_pasada_{pasada}_{c}.pth"
            
        ruta = os.path.join(path_modelos, nombre)

        if not os.path.exists(ruta):
            print(f"Sin modelo para {c}: {ruta}")
            continue

        # Usamos weights_only=False por compatibilidad con versiones recientes de PyTorch
        checkpoint = torch.load(ruta, map_location=device, weights_only=False)

        m = Conv1DAutoencoder(
            input_channels=input_channels, 
            seq_len=seq_len, 
            bottleneck_dim=bottleneck_dim
        ).to(device)
        
        m.load_state_dict(checkpoint["model_state_dict"])
        m.eval()
        modelos[c] = m
        
    return modelos

def aplanar(
    datos_por_cultivo, 
    input_channels=2, 
    seq_len=61
    ):
    """
    Transforma un diccionario estructurado por cultivos en tensores y arreglos planos 
    listos para ser procesados por el modelo.
    """
    lista_datos = []
    lista_etiquetas = []
    lista_pid = []

    for clase, datos in datos_por_cultivo.items():
        # 1. Procesar las características (X): convertir a tensor y darles forma de red neuronal
        #print("datos[X]:",datos["X"])
        x_tensor = torch.as_tensor(datos["X"], dtype=torch.float32)
        #print(x_tensor)
        x_tensor = x_tensor.reshape(-1, input_channels, seq_len)
        #print(x_tensor)
        lista_datos.append(x_tensor)
        #print("lista:",lista_datos)

        # 2. Crear las etiquetas (Y): repetir el nombre de la clase tantas veces como muestras tenga
        num_muestras = len(x_tensor)
        etiquetas_clase = [clase] * num_muestras
        lista_etiquetas.extend(etiquetas_clase)

        # 3. Guardar los identificadores de parcelas
        parcelas_array = np.asarray(datos["parcelas"])
        lista_pid.append(parcelas_array)

    # Unir todas las listas de los diferentes cultivos en una sola estructura final
    X_final = torch.cat(lista_datos)
    Y_final = np.array(lista_etiquetas)
    pid_final = np.concatenate(lista_pid)

    return X_final, Y_final, pid_final

def predict_pixels(
    datos_pixeles,
    etiquetas_reales,
    modelos_cae,
    batch_size=4096,
    device="cuda"
):
    """
    Evalúa la fiabilidad de cada píxel utilizando los CAE expertos.

    Para cada píxel:
        1. Calcula el error de reconstrucción de cada CAE.
        2. Identifica el CAE con menor error.
        3. Compara la clase candidata con la etiqueta real.
        4. Determina si el píxel es fiable o sospechoso.

    Devuelve:
        - matriz_errores: errores MSE de cada píxel y cada CAE.
        - clases_candidatas: clase del CAE con menor error.
        - pixeles_fiables: máscara booleana de fiabilidad.
        - pixeles_sospechosos: máscara booleana de sospecha.
    """

    # --------------------------------------------------
    # 1. PREPARAR LOS DATOS DE ENTRADA
    # --------------------------------------------------

    if isinstance(datos_pixeles, np.ndarray):
        datos_pixeles = torch.as_tensor(
            datos_pixeles,
            dtype=torch.float32
        )

    if isinstance(etiquetas_reales, torch.Tensor):
        etiquetas_reales = etiquetas_reales.cpu().numpy()

    elif isinstance(etiquetas_reales, list):
        etiquetas_reales = np.array(etiquetas_reales)

    numero_pixeles = datos_pixeles.shape[0]

    # --------------------------------------------------
    # 2. PREPARAR LOS MODELOS EXPERTOS
    # --------------------------------------------------

    clases_modelos = sorted(modelos_cae.keys())
    numero_modelos = len(clases_modelos)

    # Una fila por píxel y una columna por CAE
    matriz_errores = np.zeros((numero_pixeles, numero_modelos),dtype=np.float32)

    funcion_mse = nn.MSELoss(reduction="none")

    # --------------------------------------------------
    # 3. EVALUAR CADA MODELO CAE
    # --------------------------------------------------

    for indice_modelo, clase_modelo in enumerate(clases_modelos):

        modelo_cae = modelos_cae[clase_modelo].to(device)
        modelo_cae.eval()

        errores_modelo = []

        with torch.no_grad():

            # Procesar los píxeles por lotes
            for inicio in range(0, numero_pixeles, batch_size):

                fin = min(inicio + batch_size, numero_pixeles)

                lote_pixeles = datos_pixeles[inicio:fin].to(device)

                # Reconstrucción mediante el CAE
                pixeles_reconstruidos = modelo_cae(lote_pixeles)

                # Calcular MSE individual por píxel
                errores_lote = funcion_mse(
                    lote_pixeles,
                    pixeles_reconstruidos
                ).mean(dim=(1, 2))

                errores_modelo.append(
                    errores_lote.cpu().numpy()
                )

        # Unir los errores de todos los lotes
        errores_modelo = np.concatenate(errores_modelo)

        # Guardar los errores en la columna del modelo actual
        matriz_errores[:, indice_modelo] = errores_modelo

    # --------------------------------------------------
    # 4. IDENTIFICAR EL CAE CON MENOR ERROR POR PÍXEL
    # --------------------------------------------------

    indices_mejor_modelo = np.argmin(
        matriz_errores,
        axis=1
    )

    clases_candidatas = np.array(clases_modelos)[
        indices_mejor_modelo
    ]

    # --------------------------------------------------
    # 5. DETERMINAR LA FIABILIDAD DE CADA PÍXEL
    # --------------------------------------------------

    pixeles_fiables = (
        clases_candidatas == etiquetas_reales
    )

    pixeles_sospechosos = ~pixeles_fiables

    # --------------------------------------------------
    # 6. DEVOLVER LOS RESULTADOS
    # --------------------------------------------------

    return {
        "mse_matrix": matriz_errores,
        "candidate_classes": clases_candidatas,
        "is_trustworthy": pixeles_fiables,
        "is_suspicious": pixeles_sospechosos
    }

def parcelas_fiables(X, Y, pid, modelos, device, umbral=UMBRAL):
    """Devuelve (ids_fiables, clase_de_cada_parcela_fiable)."""
    pixeles_fiables = predict_pixels(X, Y, modelos, batch_size=4096, device=device)["is_trustworthy"]
    upid, first, inv = np.unique(pid, return_index=True, return_inverse=True)
    frac_ok = np.bincount(inv, weights=pixeles_fiables.astype(float)) / np.bincount(inv)
    sel = frac_ok > umbral
    return upid[sel], Y[first][sel]

def infectar(clase_parcela, pct, rng):
    """
    Permuta las etiquetas de pct % de las parcelas, conservando la
    distribución de clases. Ninguna parcela elegida conserva su etiqueta
    (salvo que sea imposible). Devuelve las etiquetas infectadas.
    """
    n = len(clase_parcela)
    k = int(round(n * pct / 100))
    nuevas = clase_parcela.copy()
    if k < 2:
        return nuevas
    idx = rng.choice(n, size=k, replace=False)
    orig = clase_parcela[idx]
    perm = rng.permutation(orig)
    for _ in range(5):
        iguales = np.where(perm == orig)[0]
        if len(iguales) == 0:
            break
        for i in iguales:
            if perm[i] != orig[i]:
                continue
            cand = np.where((perm != orig[i]) & (orig != perm[i]))[0]
            if len(cand) == 0:
                continue
            j = rng.choice(cand)
            perm[i], perm[j] = perm[j], perm[i]
    nuevas[idx] = perm
    return nuevas

def construir_dataset_infectado(X, pid, ids_fiables, etiquetas_inf):
    """Se queda con los píxeles de parcelas fiables y les asigna la etiqueta infectada."""
    orden = np.argsort(ids_fiables)
    ids_s, et_s = ids_fiables[orden], etiquetas_inf[orden]
    pos = np.clip(np.searchsorted(ids_s, pid), 0, len(ids_s) - 1)
    valido = ids_s[pos] == pid
    return X[torch.as_tensor(valido)], pid[valido], et_s[pos[valido]]

def generar_datasets_infectados(X, y, pid, ids_fiables, clase_fiable, dir_salida,
                                niveles=(1, 5, 10, 15, 20, 25, 30), n_rep=10):
    """
    Escribe dataset_noise_{pct}pct_rep_{rep}.pt con el formato que lee
    entrenamiento_ruidoso_farmsar.py:
        {"train": {clase_infectada: {"X", "parcelas", "y_original"}},
         "meta":  {pct, rep, ids_fiables, clase_fiable, etiqueta_infectada}}
    y_original (por píxel) y meta permiten calcular recall y precisión después.
    Cada archivo pesa lo que pesa el conjunto fiable: comprueba el espacio en disco.
    """
    os.makedirs(dir_salida, exist_ok=True)
    valido = np.isin(pid, ids_fiables)
    y_orig = y[valido]
    for pct in niveles:
        for rep in range(n_rep):
            ruta = os.path.join(dir_salida, f"dataset_noise_{pct:02d}pct_rep_{rep:02d}.pt")
            if os.path.exists(ruta):
                continue
            rng = np.random.default_rng(1000 * pct + rep)
            inf = infectar(clase_fiable, pct, rng)
            Xf, pf, yinf = construir_dataset_infectado(X, pid, ids_fiables, inf)
            datos = {}
            for c in np.unique(yinf):
                m = yinf == c
                datos[str(c)] = {
                    "X": Xf[torch.as_tensor(m)],
                    "parcelas": pf[m],
                    "y_original": y_orig[m],
                }
            torch.save({
                "train": datos,
                "meta": {"pct": pct, "rep": rep, "ids_fiables": ids_fiables,
                         "clase_fiable": clase_fiable, "etiqueta_infectada": inf},
            }, ruta)
            print(f"Guardado {ruta} (infectadas reales: {int((inf != clase_fiable).sum())})")

def metricas(orig, inf, final, relabel):
    """
    recall    = infectadas devueltas a su clase original / infectadas
    precision = relabels correctos / relabels totales
    (un relabel sobre una parcela no infectada cuenta como error)
    """
    infectada = inf != orig
    correcto = relabel & infectada & (final == orig)
    return {
        "recall": correcto.sum() / max(infectada.sum(), 1),
        "precision": correcto.sum() / max(relabel.sum(), 1),
        "n_infectadas": int(infectada.sum()),
        "n_relabels": int(relabel.sum()),
    }


# Ejemplo de uso:
# X, y, pid = aplanar_test(data["test"])
# ids_f, cls_f = parcelas_fiables(X, y, pid, modelos_finales, device)
# kw = dict(input_channels=2, seq_len=61, bottleneck_dim=1, batch_size=128, epochs=20, lr=1e-3)
# resultados = {}
# for pct in [1, 5, 10, 15, 20, 25, 30]:
#     resultados[pct] = [un_experimento(X, y, pid, ids_f, cls_f, pct, s, device, **kw) for s in range(10)]

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'


CLASES_MAESTRO = {
    "Alfalfa":     1,
    "Barley":      2,
    "Carrot":      3,
    "Chickpea":    4,
    "Cotton":      5,
    "Fallow":      6,
    "Maize":       7,
    "Oats":        8,
    "Onion":       9,
    "Pepper":      10,
    "Potato":      11,
    "Pumpkin":     12,
    "Quinoa":      13,
    "Rice":        14,
    "SugarBeet":   15,
    "Sunflower":   16,
    "SweetPotato": 17,
    "Tomato":      18,
    "Wheat":       19
}

data = torch.load(
    "/home/angelo/MyStore/Proyecto Investigador/2017/dataset_split_2017/dataset_split_2017_(backscatter)_v1.pt",
    weights_only=False
)

X, y, pid = aplanar(data["test"])
path_modelos = "/home/angelo/MyStore/Proyecto Investigador/2017/modelos_CAE/Modelos_temporales_2017_FARMSAR_BOTTLENECK_DIM_1_10pass_v1/"
modelos_finales = cargar_modelos(path_modelos, CLASES_MAESTRO.keys(), pasada=10, input_channels=2, seq_len=61, bottleneck_dim=1, device=device)
ids_f, cls_f = parcelas_fiables(X, y, pid, modelos_finales, device)

print(len(ids_f), "parcelas fiables de", len(np.unique(pid)))
print(dict(zip(*np.unique(cls_f, return_counts=True))))   # fiables por clase

Sin modelo para Barley: /home/angelo/MyStore/Proyecto Investigador/2017/modelos_CAE/Modelos_temporales_2017_FARMSAR_BOTTLENECK_DIM_1_10pass_v1/cae_pasada_10_Barley.pth
Sin modelo para Oats: /home/angelo/MyStore/Proyecto Investigador/2017/modelos_CAE/Modelos_temporales_2017_FARMSAR_BOTTLENECK_DIM_1_10pass_v1/cae_pasada_10_Oats.pth
Sin modelo para Rice: /home/angelo/MyStore/Proyecto Investigador/2017/modelos_CAE/Modelos_temporales_2017_FARMSAR_BOTTLENECK_DIM_1_10pass_v1/cae_pasada_10_Rice.pth
1240 parcelas fiables de 1777
{np.str_('Alfalfa'): np.int64(32), np.str_('Carrot'): np.int64(22), np.str_('Chickpea'): np.int64(1), np.str_('Cotton'): np.int64(575), np.str_('Fallow'): np.int64(10), np.str_('Maize'): np.int64(66), np.str_('Onion'): np.int64(10), np.str_('Pepper'): np.int64(1), np.str_('Potato'): np.int64(1), np.str_('Pumpkin'): np.int64(4), np.str_('Quinoa'): np.int64(7), np.str_('SugarBeet'): np.int64(224), np.str_('Sunflower'): np.int64(49), np.str_('SweetPotato'): np.int64(5), np

In [9]:
"""
Fiables en el  conjunto de test.

  * Fiables según CAE : más del 75 % de los píxeles de la parcela tienen como CAE
                        de mínimo error el de su propia clase. Se calculan sobre
                        train con los CAE de la primera etapa (en muestra, como en
                        el artículo).
  * Fiables según RF  : más del 75 % de los píxeles de la parcela se predicen con su
                        etiqueta original, en validación cruzada de 4 folds POR
                        PARCELA dentro de train. No se puede usar un RF ya entrenado
                        con train: al aplicarlo a train estaría prediciendo parcelas
                        que ya vio (memorización).
  * Intersección      : parcelas aceptadas por ambos.

"""

RUTA_SALIDA= ("/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/fiables_interseccion_v1.npz")
RUTA_RF = ("/home/angelo/MyStore/Proyecto Investigador/2017/modelos_RF/rf_modelo_(backscatter)_v1.joblib")

def _limitar_por_parcela(idx, pid, max_pix, rng):
    """De los índices idx, conserva como mucho max_pix píxeles al azar por parcela."""
    idx = idx[rng.permutation(len(idx))]
    _, inv = np.unique(pid[idx], return_inverse=True)
    orden = np.argsort(inv, kind="stable")
    inv_ord = inv[orden]
    inicio = np.r_[0, np.flatnonzero(np.diff(inv_ord)) + 1]
    rango = np.arange(len(inv_ord)) - inicio[inv_ord]
    return idx[orden[rango < max_pix]]


def rf_fiables_oof(Xf, y, pid, semilla=0, umbral=UMBRAL, max_pix_parcela=None):
    """
    RF fuera de fold por parcela. Xf: (N,122). Devuelve upid, clase_parcela, frac, ids_rf.

    max_pix_parcela: si se indica (por ejemplo 100), cada RF se entrena con como mucho
    esos píxeles por parcela (más rápido). La predicción se hace siempre sobre TODOS
    los píxeles de las parcelas del fold de test. None = entrenar con todos.
    """
    oof = np.empty(len(y), dtype=y.dtype)
    cv = StratifiedGroupKFold(n_splits=N_FOLDS_FIABLES, shuffle=True, random_state=semilla)
    for k, (tr, te) in enumerate(cv.split(Xf, y, groups=pid), 1):
        assert np.intersect1d(np.unique(pid[tr]), np.unique(pid[te])).size == 0, \
            "Una parcela está en train y test del mismo fold"
        if max_pix_parcela:
            tr = _limitar_por_parcela(tr, pid, max_pix_parcela, np.random.default_rng(semilla + k))
        print(f"  fold {k}/{N_FOLDS_FIABLES}: entrenando un RF nuevo con {len(tr)} píxeles "
              f"(parcelas del fold {k} apartadas)...")
        rf = RandomForestClassifier(**RF_PARAMS_FIABLES, random_state=semilla)
        rf.fit(Xf[tr], y[tr])
        oof[te] = rf.predict(Xf[te])
    upid, first, inv = np.unique(pid, return_index=True, return_inverse=True)
    frac = np.bincount(inv, weights=(oof == y).astype(float)) / np.bincount(inv)
    return upid, y[first], frac, upid[frac > umbral]


def rf_fiables_modelo(
    rf,
    Xf,
    y,
    pid,
    umbral=UMBRAL,
    mapa_clases=None
):
    """
    Calcula la fiabilidad de las parcelas mediante un RF
    previamente entrenado.

    Una parcela es fiable si más del 75 % de sus píxeles
    son clasificados con su etiqueta original.
    """

    y = np.asarray(y)
    pid = np.asarray(pid)

    assert len(Xf) == len(y) == len(pid)

    if rf.n_features_in_ != Xf.shape[1]:
        raise ValueError("Dimensión de características incompatible.")

    clases_rf = np.asarray(rf.classes_)
    y_cmp = y.copy()

    # Si el RF utiliza etiquetas numéricas y y contiene nombres
    if clases_rf.dtype.kind in "iuf":

        etiquetas = np.unique(y)

        son_nombres = all(
            isinstance(c, (str, np.str_))
            for c in etiquetas
        )

        if son_nombres:

            if mapa_clases is None:
                raise ValueError(
                    "El RF utiliza etiquetas numéricas. "
                    "Debes proporcionar mapa_clases."
                )

            faltan = [
                c for c in etiquetas
                if c not in mapa_clases
            ]

            if faltan:
                raise ValueError(
                    f"Faltan clases en mapa_clases: {faltan}"
                )

            y_cmp = np.array(
                [mapa_clases[c] for c in y]
            )

    # Comprobar compatibilidad de las clases
    conocidas = (
        set(np.unique(y_cmp).tolist())
        & set(clases_rf.tolist())
    )

    if not conocidas:
        raise ValueError(
            "Las etiquetas de TEST no coinciden con "
            "las etiquetas utilizadas por el RF."
        )

    sin_clase = (
        set(np.unique(y_cmp).tolist())
        - set(clases_rf.tolist())
    )

    if sin_clase:
        print(
            "Aviso: el RF no conoce estas clases:",
            sin_clase
        )

    # Predicción de todos los píxeles de TEST
    pred = rf.predict(Xf)

    # Agrupar los píxeles por parcela
    upid, first, inv = np.unique(
        pid,
        return_index=True,
        return_inverse=True
    )

    # Proporción de píxeles correctamente clasificados
    correctos = (pred == y_cmp).astype(float)

    frac = (
        np.bincount(inv, weights=correctos)
        / np.bincount(inv)
    )

    # Seleccionar parcelas con más del umbral establecido
    ids_rf = upid[frac > umbral]

    return upid, y[first], frac, ids_rf


def resumen_fiables(upid, clase_parcela, ids_cae, ids_rf):
    ids_inter = np.intersect1d(ids_cae, ids_rf)
    solo_cae = np.setdiff1d(ids_cae, ids_rf)
    solo_rf = np.setdiff1d(ids_rf, ids_cae)
    print("\n" + "=" * 72)
    print(f"Parcelas evaluadas:            {len(upid)}")
    print(f"Fiables según CAE:            {len(ids_cae)}")
    print(f"Fiables según RF:             {len(ids_rf)}")
    print(f"Intersección:                 {len(ids_inter)}")
    print(f"Solo CAE:                     {len(solo_cae)}")
    print(f"Solo RF:                      {len(solo_rf)}")
    print(f"De las fiables CAE, aceptadas también por RF: "
          f"{100 * len(ids_inter) / max(len(ids_cae), 1):.1f} %")
    print("=" * 72)
    print(f"\n{'clase':<14}{'total':>6}{'CAE':>6}{'RF':>6}{'inter.':>8}{'solo CAE':>10}")
    for c in sorted(np.unique(clase_parcela)):
        ids_c = upid[clase_parcela == c]
        n_int = np.isin(ids_c, ids_inter).sum()
        aviso = "   <-- pocas" if n_int < N_MIN_PARCELAS else ""
        print(f"{c:<14}{len(ids_c):>6}{np.isin(ids_c, ids_cae).sum():>6}{np.isin(ids_c, ids_rf).sum():>6}"
              f"{n_int:>8}{np.isin(ids_c, solo_cae).sum():>10}{aviso}")
    return ids_inter, solo_cae, solo_rf

def filtrar_clases_pocas(ids_inter, cls_inter, n_min=N_MIN_PARCELAS, clases_validas=None):
        clases, n = np.unique(cls_inter, return_counts=True)
        if clases_validas is None:
            clases_validas = clases[n >= n_min]
        validas = set(np.asarray(clases_validas).tolist())
        mask = np.isin(cls_inter, list(validas))
        excluidas = {c: int(k) for c, k in zip(clases, n) if c not in validas}
        return ids_inter[mask], cls_inter[mask], np.asarray(clases_validas), excluidas

def calcular_interseccion_test(
    data,
    modelos_finales,
    device,
    ruta_salida=RUTA_SALIDA,
    ids_cae=None,
    rf_previo=RUTA_RF,
    mapa_clases=None,
    n_min=N_MIN_PARCELAS, 
    clases_validas=None
):
    """
    Calcula las parcelas fiables según CAE y RF sobre TEST.

    Ambos modelos deben estar previamente entrenados con TRAIN.

    CAE:
        Se utilizan los modelos ya entrenados para reconstruir TEST.

    RF:
        Se utiliza el Random Forest ya entrenado para predecir TEST.

    Finalmente:
        Se calcula la intersección de parcelas fiables.
    """

    # --------------------------------------------------
    # 1. Comprobar que existe la ruta de salida
    # --------------------------------------------------

    if os.path.exists(ruta_salida):
        print(f"⚠ {ruta_salida} ya existe. Se sobreescribirá.")

    # --------------------------------------------------
    # 2. Obtener los datos de TEST
    # --------------------------------------------------

    X, y, pid = aplanar(data["test"])

    print("\n" + "=" * 65)
    print("EVALUACIÓN DE PARCELAS FIABLES SOBRE TEST")
    print("=" * 65)

    print(f"Píxeles de test: {len(X)}")
    print(f"Parcelas de test: {len(np.unique(pid))}")
    print(f"Clases: {len(np.unique(y))}")

    assert len(X) == len(y) == len(pid), (
        "X, y y pid deben tener el mismo número de muestras."
    )

    # --------------------------------------------------
    # 3. Fiabilidad según CAE
    # --------------------------------------------------

    if ids_cae is None:

        if modelos_finales is None:
            raise ValueError(
                "Debes proporcionar los modelos CAE entrenados "
                "o los identificadores ids_cae."
            )

        print("\nCalculando parcelas fiables según CAE...")

        ids_cae, clases_cae = parcelas_fiables(
            X,
            y,
            pid,
            modelos_finales,
            device
        )

    else:

        ids_cae = np.unique(
            np.asarray(ids_cae).reshape(-1)
        )

        fuera = np.setdiff1d(ids_cae, np.unique(pid))

        if fuera.size > 0:
            raise ValueError(
                f"{len(fuera)} parcelas de ids_cae no pertenecen a TEST."
            )

        print(
            f"\nSe utilizan {len(ids_cae)} parcelas "
            "fiables según CAE previamente calculadas."
        )

    # --------------------------------------------------
    # 4. Preparar características para RF
    # --------------------------------------------------

    Xf = X.detach().cpu().numpy().reshape(len(X), -1)

    print(f"\nDimensiones de X para RF: {Xf.shape}")

    # --------------------------------------------------
    # 5. Cargar el Random Forest ya entrenado
    # --------------------------------------------------

    if isinstance(rf_previo, (str, os.PathLike)):
        contenido = joblib.load(rf_previo)
        if isinstance(contenido, dict):
            rf = contenido["model"]
            print(f"RF cargado: año={contenido.get('año','?')} | "
                f"features={contenido.get('input_shape','?')}")
        else:
            rf = contenido  # compatibilidad con modelos guardados sin dict
    else:
        rf = rf_previo
        print("\nRandom Forest cargado correctamente.")
        print(f"Características esperadas: {rf.n_features_in_}")

    # --------------------------------------------------
    # 6. Fiabilidad según RF
    # --------------------------------------------------

    print("\nCalculando parcelas fiables según RF...")

    upid, clase_parcela, frac_rf, ids_rf = rf_fiables_modelo(
        rf,
        Xf,
        y,
        pid,
        mapa_clases=mapa_clases
    )

    # --------------------------------------------------
    # 7. Intersección CAE - RF
    # --------------------------------------------------

    ids_inter, solo_cae, solo_rf = resumen_fiables(
        upid,
        clase_parcela,
        ids_cae,
        ids_rf
    )

    # Recuperar la clase original de cada parcela de la intersección
    cls_inter = clase_parcela[np.isin(upid, ids_inter)]

    assert len(cls_inter) == len(ids_inter)
    
    ids_inter_total, cls_inter_total = ids_inter, cls_inter
    ids_inter, cls_inter, clases_validas, excluidas = filtrar_clases_pocas(
        ids_inter, cls_inter, n_min=n_min, clases_validas=clases_validas)
    print(f"Clases excluidas (< {n_min} parcelas en la intersección): {excluidas}")
       
    
    # --------------------------------------------------
    # 8. Guardar resultados
    # --------------------------------------------------

    directorio = os.path.dirname(ruta_salida)

    if directorio:
        os.makedirs(directorio, exist_ok=True)

    np.savez_compressed(
        ruta_salida,

        ids_cae=ids_cae,
        ids_rf=ids_rf,
        ids_inter=ids_inter,

        ids_inter_total=ids_inter_total, 
        cls_inter_total=cls_inter_total, 
        clases_validas=clases_validas,
        
        cls_inter=cls_inter,

        ids_solo_cae=solo_cae,
        ids_solo_rf=solo_rf,

        upid=upid,
        clase_parcela=clase_parcela,

        frac_rf=frac_rf,

        mitad=np.array("test"),
        metodo_rf=np.array("modelo_previamente_entrenado")
        
        
    )

    print("\n" + "=" * 65)
    print("RESULTADOS GUARDADOS")
    print("=" * 65)
    print(f"Archivo: {ruta_salida}")
    print(f"Parcelas fiables CAE: {len(ids_cae)}")
    print(f"Parcelas fiables RF: {len(ids_rf)}")
    print(f"Intersección: {len(ids_inter)}")

    return {
        "X": X,
        "y": y,
        "pid": pid,

        "ids_cae": ids_cae,
        "ids_rf": ids_rf,
        "ids_inter": ids_inter,

        "cls_inter": cls_inter,

        "upid": upid,
        "clase_parcela": clase_parcela,

        "frac_rf": frac_rf
    }

In [10]:
res = calcular_interseccion_test(
    data=data,
    modelos_finales=modelos_finales,
    device=device,
    ruta_salida=RUTA_SALIDA,
    ids_cae=ids_f,
    rf_previo=RUTA_RF,
    mapa_clases=CLASES_MAESTRO,
    n_min=N_MIN_PARCELAS, 
    clases_validas=None
)

⚠ /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/fiables_interseccion_v1.npz ya existe. Se sobreescribirá.

EVALUACIÓN DE PARCELAS FIABLES SOBRE TEST
Píxeles de test: 568967
Parcelas de test: 1777
Clases: 16

Se utilizan 1240 parcelas fiables según CAE previamente calculadas.

Dimensiones de X para RF: (568967, 122)
RF cargado: año=2017 | features=122

Calculando parcelas fiables según RF...

Parcelas evaluadas:            1777
Fiables según CAE:            1240
Fiables según RF:             1592
Intersección:                 1226
Solo CAE:                     14
Solo RF:                      366
De las fiables CAE, aceptadas también por RF: 98.9 %

clase          total   CAE    RF  inter.  solo CAE
Alfalfa           40    32    34      31         1
Carrot            36    22    21      19         3
Chickpea          14     1     5       1         0   <-- pocas
Cotton           662   575   625     571         4
Fallow            37    10    16       9 

In [12]:
dir_base_datasets = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1"
generar_datasets_infectados(
    res["X"],
    res["y"],
    res["pid"],
    res["ids_inter"],
    res["cls_inter"],
    dir_base_datasets,
    niveles=[1, 5, 10, 15, 20, 25, 30],
    n_rep=10
)

Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/dataset_noise_01pct_rep_00.pt (infectadas reales: 10)
Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/dataset_noise_01pct_rep_01.pt (infectadas reales: 12)
Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/dataset_noise_01pct_rep_02.pt (infectadas reales: 12)
Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/dataset_noise_01pct_rep_03.pt (infectadas reales: 6)
Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/dataset_noise_01pct_rep_04.pt (infectadas reales: 10)
Guardado /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1